# 🚀 SycophancyRAGBench: Master One-Cell Execution (A100 Optimized)
### *Multi-Paradigm Benchmark, Causal Conflict Analysis & Conflict-Aware Epistemic Verification (CAEV)*

> **Key Capabilities:**
> - 🏢 **Multi-Scale & Multi-Family Models:** Qwen 2.5 (3B), Gemma 3 (4B), Llama 3.2 (3B), and **Llama 3.1 (8B)**
> - 🔍 **Multi-Paradigm Retrieval:** Sparse (BM25), Dense (BGE), Hybrid (RRF)
> - 🛡️ **Conflict-Aware Epistemic Verification (CAEV):** `rag_hybrid_caev` & `oracle_caev` mitigation conditions
> - ⚖️ **Experimental Controls:** Oracle Evidence (upper bound) & Irrelevant Distractors (noise floor)
> - 💾 **Verified Google Drive Mount & Live Checkpointing (`FRESH_RUN = False`):** Verifies `/content/drive` is genuinely mounted to Google Drive (`os.path.ismount`), recovers any files written to unmounted local `/content/` folders, and flushes every batch live to `My Drive/SycophancyRAGBench`!


## ⚡ Quickstart Instructions (Google Colab A100)

1. Upload `sycophancy_rag_bench.zip` to `/content/` in Google Colab (or place it in your Google Drive root `MyDrive/sycophancy_rag_bench.zip`).
2. Set Runtime to **GPU -> A100**.
3. Run the cell below, approve the Google Drive popup, and paste your Hugging Face Token when prompted.
4. All checkpoints, raw generations, judge evaluations, figures, tables, and the final `sycophancy_rag_bench_outputs.zip` are saved directly in your Google Drive at `My Drive/SycophancyRAGBench`.


In [ ]:
# ==============================================================================
# 🚀 ONE-CELL UNIVERSAL RUNNER (Google Colab A100 & Kaggle Notebooks)
# ==============================================================================
import os, sys, shutil

def _verify_colab_drive_mount():
    if not os.path.exists('/content'):
        return
    print('[Platform] Detected Google Colab.')
    from google.colab import drive
    is_mounted = os.path.ismount('/content/drive') or os.path.islink('/content/drive/MyDrive')
    backup_dir = '/content/_unmounted_drive_backup'
    if not is_mounted:
        if os.path.exists('/content/drive') and os.listdir('/content/drive'):
            print('[Drive] Found unmounted local /content/drive folder! Backing it up before mounting real Google Drive...')
            if os.path.exists(backup_dir):
                shutil.rmtree(backup_dir)
            shutil.move('/content/drive', backup_dir)
            os.makedirs('/content/drive', exist_ok=True)
        print('[Drive] Mounting real Google Drive to /content/drive (please approve popup)...')
        drive.mount('/content/drive', force_remount=True)

    target_drive_dir = '/content/drive/MyDrive/SycophancyRAGBench'
    os.makedirs(target_drive_dir, exist_ok=True)
    # Recover any files from previous unmounted local runs into real Google Drive
    for local_candidate in [os.path.join(backup_dir, 'MyDrive', 'SycophancyRAGBench'), '/content/SycophancyRAGBench']:
        if os.path.exists(local_candidate):
            print(f'[Drive] Recovering previous checkpoints from {local_candidate} -> {target_drive_dir} ...')
            shutil.copytree(local_candidate, target_drive_dir, dirs_exist_ok=True)
    print(f'[Drive] Verified Google Drive folder ready: {target_drive_dir}')

# 1. Mount & Verify Google Drive first on Colab
if os.path.exists('/content'):
    _verify_colab_drive_mount()
    zip_candidates = [
        '/content/sycophancy_rag_bench.zip',
        '/content/drive/MyDrive/sycophancy_rag_bench.zip',
        '/content/drive/MyDrive/SycophancyRAGBench/sycophancy_rag_bench.zip'
    ]
    found_zip = next((z for z in zip_candidates if os.path.exists(z)), None)
    if found_zip:
        print(f'[Setup] Extracting {found_zip} ...')
        !unzip -q -o "{found_zip}" -d /content/
elif os.path.exists('/kaggle/working'):
    print('[Platform] Detected Kaggle Notebooks.')
    if os.path.exists('/kaggle/input/sycophancy-rag-bench/sycophancy_rag_bench.zip'):
        !unzip -q -o /kaggle/input/sycophancy-rag-bench/sycophancy_rag_bench.zip -d /kaggle/working/
    elif os.path.exists('/kaggle/working/sycophancy_rag_bench.zip'):
        !unzip -q -o /kaggle/working/sycophancy_rag_bench.zip -d /kaggle/working/

pkg_candidates = [
    '/content/sycophancy_rag_bench',
    '/kaggle/working/sycophancy_rag_bench',
    os.path.join(os.getcwd(), 'sycophancy_rag_bench'),
    os.getcwd()
]
PKG_DIR = next((p for p in pkg_candidates if os.path.exists(os.path.join(p, 'run_all.py'))), os.getcwd())
if PKG_DIR not in sys.path:
    sys.path.insert(0, PKG_DIR)
os.chdir(PKG_DIR)
print(f'[Setup] Working directory set to: {PKG_DIR}')

# 2. Install dependencies (quietly)
print('[Setup] Verifying and installing dependencies...')
!pip install -q -U "transformers>=4.51.0" accelerate bitsandbytes sentence-transformers datasets statsmodels scikit-learn matplotlib tqdm

# 3. Benchmark Run Configuration
MODELS_TO_RUN = ['qwen2.5-3b', 'gemma-3-4b', 'llama-3.2-3b', 'llama-3.1-8b']
SETTINGS_TO_RUN = [
    'vanilla', 'rag_bm25', 'rag_dense', 'rag_hybrid',
    'control_oracle', 'control_irrelevant',
    'rag_hybrid_caev', 'oracle_caev'
]

# 🔑 Hugging Face Authentication Token:
# Paste your token here (e.g. "hf_..."), or leave as None to be prompted interactively:
HF_TOKEN = None

# 📊 Dataset Scale Configuration:
TAKE_ALL_DATA = False
SAMPLE_LIMIT = None

# 🔄 Checkpoint & Incremental Resume Configuration:
# Keep FRESH_RUN = False so all checkpoints on Google Drive and precomputed baselines are preserved and resumed!
FRESH_RUN = False

# 4. Execute the End-to-End Pipeline
from run_all import run_pipeline

class ConfigArgs:
    models = MODELS_TO_RUN
    settings = SETTINGS_TO_RUN
    fresh = FRESH_RUN
    all_data = TAKE_ALL_DATA
    sample_limit = SAMPLE_LIMIT
    batch_size = 16
    hf_token = HF_TOKEN
    skip_retrieval = False
    skip_inference = False
    skip_eval = False
    skip_stats = False
    zip_on_finish = True

print('\n' + '='*60)
print('STARTING SYCOPHANCYRAGBENCH BENCHMARK EXECUTION...')
print(f'Models:   {MODELS_TO_RUN}')
print(f'Settings: {SETTINGS_TO_RUN}')
print(f'Fresh:    {FRESH_RUN} (False = Resume from Drive checkpoints & precomputed baselines)')
print(f'Sample:   {SAMPLE_LIMIT if SAMPLE_LIMIT else "FULL (1,800 prompts)"}')
print('='*60 + '\n')

run_pipeline(ConfigArgs())

# 5. Display Generated Publication Figures Inline
from IPython.display import Image, display
from configs import config as cfg

fig1 = os.path.join(cfg.FIG_DIR, 'sycophancy_by_model_and_retriever.png')
fig2 = os.path.join(cfg.FIG_DIR, 'evidence_user_conflict_breakdown.png')

if os.path.exists(fig1):
    print('\n--- Generated Figure 1: Sycophancy by Model & Retriever ---')
    display(Image(fig1))

if os.path.exists(fig2):
    print('\n--- Generated Figure 2: Causal Evidence-User Conflict Breakdown ---')
    display(Image(fig2))
